# Final recorded-outcome evaluation

Fit the agreed classifier on 2019–2025 bills and evaluate it on 2026 bills.
Choose its inputs and settings from [notebook 06](06_model_evaluation.ipynb)
before viewing the 2026 scores. This is retrospective classification of the
published outcome labels, not prediction at introduction.

The configuration is initially unselected. Running this notebook in that state
does not load evaluation data, fit a model or produce test results.

## 1. Setup

In [1]:
import hashlib
import json
import sys
from importlib.metadata import version
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from sklearn.metrics import (
    ConfusionMatrixDisplay, accuracy_score, balanced_accuracy_score,
    classification_report, confusion_matrix, f1_score,
)

root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(root))
import src.classification_helpers as ch
from src.topic_analysis import LEGAL_STOPWORDS, VECTOR_SETTINGS

LABELS = ["Failed", "Passed", "Pending"]
TARGET = "Bill_Status_Grouped"
FEATURE_GROUPS = {
    "categorical": ["measure_code", "Jurisdiction", "State_Political_Control"],
    "numeric": ["D_Count", "R_Count", "Other_Count", "Unknown_Count", "Total_Sponsors"],
    "ratio": ["Democrat_Sponsor_Ratio", "Republican_Sponsor_Ratio"],
    "binary": ["Bipartisan_Support"],
}
FEATURE_SETS = {
    "metadata": None, "summary": "NCSL_Summary_clean", "bill_text": "Bill_Text_clean",
}
TFIDF_PARAMS = {
    "max_features": 5000, "ngram_range": (1, 2), "min_df": 2,
    "max_df": 0.95, "sublinear_tf": True, "stop_words": sorted(LEGAL_STOPWORDS),
}
summary_tfidf = {
    **VECTOR_SETTINGS,
    "stop_words": sorted(set(ENGLISH_STOP_WORDS) | set(LEGAL_STOPWORDS)),
}
text_only = {group: [] for group in FEATURE_GROUPS}
data_path = root / "data/processed/common_intake/common_records.csv"
output_dir = root / "data/processed/final_evaluation"

## 2. Record the agreed configuration

Replace `None` with the chosen configuration dictionary from notebook 06.
For example, a metadata SVM uses `model_type="linear_svm"`,
`feature_set="metadata"` and a `model_params` dictionary. Keep the agreed values
for class weighting, `C`, text inputs and any NMF/LDA features.

The years below define the final evaluation. Do not use its scores to choose
another configuration. A repeat run should reproduce the same agreed setup.

In [2]:
configuration = None
TRAIN_START_YEAR = 2019
TRAIN_END_YEAR = 2025
TEST_YEAR = 2026

## 3. Fit, evaluate and save the results

Preprocessing and optional topic features fit on the training bills inside the
existing pipeline. Metrics use the same three outcome labels as notebook 06.
The exports contain overall scores, per-class results, the confusion matrix,
source-row predictions and the effective settings.

The score table compares the selected model with always predicting the most
common training outcome. Both score the same test records; the reference never
uses test labels to choose its prediction.

In [3]:
def evaluate_final(config):
    """Run one selected configuration on the fixed year split."""
    common = pd.read_csv(data_path)
    years = pd.to_datetime(common["Introduction_Date"], errors="coerce").dt.year
    training = common[years.between(TRAIN_START_YEAR, TRAIN_END_YEAR)].copy()
    testing = common[years.eq(TEST_YEAR)].copy()
    if training.empty or testing.empty:
        raise ValueError("The selected training or test cohort is empty.")

    model_params = config.get("model_params", {}).copy()
    if config["model_type"] == "linear_svm":
        model_params.setdefault("max_iter", 20000)
    tfidf_params = config.get(
        "tfidf_params",
        summary_tfidf if config["feature_set"] == "summary" else TFIDF_PARAMS,
    )
    feature_groups = config.get("feature_groups", FEATURE_GROUPS)
    run = ch.run_experiment(
        model_type=config["model_type"], feature_set=config["feature_set"],
        train_df=training, val_df=testing, target=TARGET,
        feature_groups=feature_groups, feature_sets=FEATURE_SETS,
        tfidf_params=tfidf_params, model_params=model_params,
        topic_method=config.get("topic_method"), topic_components=8,
    )
    recorded, predicted = run["y_val"], run["val_predictions"]
    majority_class = training[TARGET].mode().iloc[0]
    majority_predictions = [majority_class] * len(testing)
    score_rows = []
    for name, predictions_for_score in [
        ("Selected model", predicted), ("Training majority", majority_predictions),
    ]:
        score_rows.append({
            "Model": name, "Train records": len(training), "Test records": len(testing),
            "Accuracy": accuracy_score(recorded, predictions_for_score),
            "Balanced accuracy": balanced_accuracy_score(recorded, predictions_for_score),
            "Macro F1": f1_score(
                recorded, predictions_for_score, labels=LABELS, average="macro", zero_division=0,
            ),
        })
    scores = pd.DataFrame(score_rows)
    report = classification_report(
        recorded, predicted, labels=LABELS, output_dict=True, zero_division=0,
    )
    per_class = pd.DataFrame(report).T.loc[LABELS, ["precision", "recall", "f1-score", "support"]]
    matrix = confusion_matrix(recorded, predicted, labels=LABELS)
    predictions = testing[["source_row", "Jurisdiction", "Bill_Number"]].copy()
    predictions["Recorded"] = recorded
    predictions["Predicted"] = predicted

    preprocessor = run["pipeline"].named_steps["preprocessor"]
    text_settings = None
    if config["feature_set"] != "metadata":
        text_settings = preprocessor.named_transformers_["text"].named_steps["tfidf"].get_params()
    topic_settings = None
    if config.get("topic_method") is not None:
        topics = preprocessor.named_transformers_["topics"]
        topic_settings = {
            "vectorizer": topics.named_steps["vectorizer"].get_params(),
            "model": topics.named_steps["model"].get_params(),
        }
    settings = {
        "configuration": config, "train_years": [TRAIN_START_YEAR, TRAIN_END_YEAR],
        "test_year": TEST_YEAR, "labels": LABELS, "feature_groups": feature_groups,
        "majority_reference_class": majority_class,
        "text_column": FEATURE_SETS[config["feature_set"]],
        "model_parameters": run["pipeline"].named_steps["classifier"].get_params(),
        "text_parameters": text_settings, "topic_parameters": topic_settings,
        "prepared_data_sha256": hashlib.sha256(data_path.read_bytes()).hexdigest(),
        "package_versions": {name: version(name) for name in [
            "numpy", "pandas", "scikit-learn", "xgboost",
        ]},
    }
    output_dir.mkdir(parents=True, exist_ok=True)
    scores.to_csv(output_dir / "scores.csv", index=False)
    per_class.to_csv(output_dir / "classification_report.csv", index_label="Class")
    predictions.to_csv(output_dir / "predictions.csv", index=False)
    pd.DataFrame(matrix, index=LABELS, columns=LABELS).to_csv(
        output_dir / "confusion_matrix.csv", index_label="Recorded",
    )
    (output_dir / "settings.json").write_text(
        json.dumps(settings, indent=2, default=str) + "\n", encoding="utf-8",
    )
    fig, ax = plt.subplots(figsize=(6, 5))
    ConfusionMatrixDisplay(matrix, display_labels=LABELS).plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(f"Recorded outcomes: {TEST_YEAR} evaluation")
    ax.set_xlabel("Predicted outcome")
    ax.set_ylabel("Recorded outcome")
    fig.tight_layout()
    fig.savefig(output_dir / "confusion_matrix.png", dpi=180, bbox_inches="tight")
    plt.show()
    return scores, per_class


if configuration is None:
    print("Choose the agreed configuration in section 2 before running final evaluation.")
else:
    scores, per_class = evaluate_final(configuration)
    display(scores.round(4), per_class.round(4))
    print(f"Results saved to {output_dir.relative_to(root)}")

Choose the agreed configuration in section 2 before running final evaluation.


## Exports

Results are saved in `data/processed/final_evaluation/`. These generated files
stay local until the team selects the final figures and tables for submission.
`predictions.csv` links each prediction to its original `source_row`.
`settings.json` records the configuration, fitted component parameters, input
hash and package versions needed to reproduce the run.